In [1]:
import pandas as pd

# File path
file_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/Filtered_original_space.xlsx"

# Read the Excel file
df = pd.read_excel(file_path)

# Columns to calculate mean for
columns = [
    "Na",
    "Al",
    "Si",
    "H2O",
    "Si/OH",
    "Si/Al",
    "Time (h)",
    "Si precursor sizs (nm)",
    "Temperature (°C)"
]

# Check if columns exist
missing_cols = [col for col in columns if col not in df.columns]

if missing_cols:
    print("The following columns are missing in the Excel sheet:")
    print(missing_cols)
else:
    # Calculate means
    means = df[columns].mean()

    # Print results
    print("\nMean values for selected columns:\n")
    for col, value in means.items():
        print(f"{col}: {value}")


Mean values for selected columns:

Na: 37.00782178217821
Al: 1.7386138613861388
Si: 6.337128712871285
H2O: 450.0
Si/OH: 0.17188637526970743
Si/Al: 4.121795733144743
Time (h): 71.65346534653466
Si precursor sizs (nm): 3.2475247524752477
Temperature (°C): 40.59405940594059


In [3]:
import pandas as pd
import numpy as np

# File path
file_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT.xlsx'

# Load Excel file
df = pd.read_excel(file_path)

# =========================
# Fill 'Si/Al' only if empty
# Si/Al = Si / Al
# =========================

mask_si_al_empty = df['Si/Al'].isna()

df.loc[mask_si_al_empty, 'Si/Al'] = (
    df.loc[mask_si_al_empty, 'Si'] /
    df.loc[mask_si_al_empty, 'Al']
)

# =========================
# Fill 'Si/OH' only if empty
# Si/OH = Si / Na
# =========================

mask_si_oh_empty = df['Si/OH'].isna()

df.loc[mask_si_oh_empty, 'Si/OH'] = (
    df.loc[mask_si_oh_empty, 'Si'] /
    df.loc[mask_si_oh_empty, 'Na']
)

# =========================
# Save updated file
# =========================

output_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_updated.xlsx'

df.to_excel(output_path, index=False)

print(f'Updated file saved to: {output_path}')

Updated file saved to: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_updated.xlsx


In [1]:
import pandas as pd

csv_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_Final_average_updated.csv"
output_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_updated.xlsx"

df = pd.read_csv(csv_path)
df.to_excel(output_path, index=False)

print(f"Saved: {output_path}")

Saved: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_updated.xlsx


In [2]:
import pandas as pd
from itertools import product

# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
input_path  = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_updated.xlsx"
output_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_expanded.xlsx"
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_excel(input_path)
print(f"Original shape: {df.shape}")

si_precursor_sizes = [2.0, 3.0, 3.25, 4.0, 5.0, 7.5]
cat_values = [1, 0]  # 1 = True, 0 = False

expanded_rows = []
for _, row in df.iterrows():
    for si_size, cat_1, cat_2 in product(si_precursor_sizes, cat_values, cat_values):
        new_row = row.copy()
        new_row["Si precursor sizs (nm)"] = si_size
        new_row["Pre-dissolution of Al"]  = cat_1
        new_row["Pre-dissolution of Si"]  = cat_2
        expanded_rows.append(new_row)

expanded_df = pd.DataFrame(expanded_rows).reset_index(drop=True)
expanded_df.to_excel(output_path, index=False)

print(f"Expanded shape: {expanded_df.shape}  (expected {len(df)} × 24 = {len(df)*24} rows)")
expanded_df[["Si precursor sizs (nm)", "Pre-dissolution of Al", "Pre-dissolution of Si"]].head(24)

Original shape: (31, 24)
Expanded shape: (744, 24)  (expected 31 × 24 = 744 rows)


,Si precursor sizs (nm),Pre-dissolution of Al,Pre-dissolution of Si
0,2.00,1,1
1,2.00,1,0
2,2.00,0,1
3,2.00,0,0
4,3.00,1,1
5,3.00,1,0
6,3.00,0,1
7,3.00,0,0
8,3.25,1,1
9,3.25,1,0


In [1]:
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
FILE_PATH = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/Table_S6_updated.xlsx"

# Mapping: value in Si:X  →  one-hot column name in the spreadsheet
SI_MAP = {
    "Fumed silica": "Si_source_Fumed silica",
    "Ludox-AS40":   "Si_source_Ludox-AS40",
    "Ludox-SM30":   "Si_source_Ludox-SM30",
    "TEOS":         "Si_source_TEOS",
    "Na2SiO3":      "Si_source_Na2SiO3",
}

# Mapping: value in Al:Y  →  one-hot column name in the spreadsheet
AL_MAP = {
    "Al(OH)3": "Al_source_Al(OH)3",
    "NaAlO2":  "Al_source_NaAlO2",
}
# ─────────────────────────────────────────────────────────────────────────────


def fill_one_hot(df: pd.DataFrame, source_col: str, mapping: dict) -> pd.DataFrame:
    """
    For each row, read `source_col`, then set the matching one-hot column to 1
    and all other one-hot columns in the mapping to 0.
    Unrecognised values leave all one-hot columns as 0 (and print a warning).
    """
    one_hot_cols = list(mapping.values())

    # Initialise all one-hot columns to 0
    for col in one_hot_cols:
        if col not in df.columns:
            raise ValueError(
                f"Expected column '{col}' not found in the spreadsheet. "
                f"Available columns: {df.columns.tolist()}"
            )
        df[col] = 0

    # Fill 1 where the source value matches
    for value, target_col in mapping.items():
        mask = df[source_col].astype(str).str.strip() == value
        df.loc[mask, target_col] = 1

    # Warn about unrecognised values
    known_values = set(mapping.keys())
    unrecognised = df.loc[
        ~df[source_col].astype(str).str.strip().isin(known_values), source_col
    ].dropna().unique()

    if len(unrecognised):
        print(
            f"[WARNING] Unrecognised values in '{source_col}' "
            f"(will be all-zeros): {unrecognised.tolist()}"
        )

    return df


def main():
    print(f"Reading: {FILE_PATH}")
    df = pd.read_excel(FILE_PATH, sheet_name="Sheet1")

    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}\n")

    # ── Si:X one-hot ──────────────────────────────────────────────────────────
    if "Si:X" not in df.columns:
        raise ValueError("Column 'Si:X' not found in the spreadsheet.")
    print("Filling one-hot columns for Si:X …")
    df = fill_one_hot(df, source_col="Si:X", mapping=SI_MAP)

    # ── Al:Y one-hot ──────────────────────────────────────────────────────────
    if "Al:Y" not in df.columns:
        raise ValueError("Column 'Al:Y' not found in the spreadsheet.")
    print("Filling one-hot columns for Al:Y …")
    df = fill_one_hot(df, source_col="Al:Y", mapping=AL_MAP)

    # ── Save back to the same file ────────────────────────────────────────────
    print(f"\nSaving updated file to: {FILE_PATH}")
    with pd.ExcelWriter(FILE_PATH, engine="openpyxl", mode="w") as writer:
        df.to_excel(writer, sheet_name="Sheet1", index=False)

    print("Done ✓")

    # ── Quick sanity check ────────────────────────────────────────────────────
    all_cols = list(SI_MAP.values()) + list(AL_MAP.values())
    print("\nOne-hot value counts (first occurrence per column):")
    for col in all_cols:
        counts = df[col].value_counts().to_dict()
        print(f"  {col}: {counts}")


if __name__ == "__main__":
    main()

Reading: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/Table_S6_updated.xlsx
  Shape: (38, 23)
  Columns: ['Synthesis', 'EMT', 'Na', 'Al', 'Si', 'H2O', 'Si precursor sizs (nm)', 'Si/OH', 'Si/Al', 'Time ', 'Temperature (°C)', 'Si:X', 'Al:Y', 'Source', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40', 'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS', 'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2', 'Pre-dissolution of Al']

Filling one-hot columns for Si:X …
Filling one-hot columns for Al:Y …

Saving updated file to: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/Table_S6_updated.xlsx
Done ✓

One-hot value counts (first occurrence per column):
  Si_source_Fumed silica: {0: 37, 1: 1}
  Si_source_Ludox-AS40: {0: 33, 1: 5}
  Si_source_Ludox-SM30: {0: 37, 1: 1}
  Si_source_TEOS: {0: 38}
  Si_source_Na2SiO3: {1: 31, 0: 7}
  Al_source_Al(OH)3: {0: 37, 1: 1}
  Al_source_NaAlO2: {1: 37, 0: 1}


In [2]:
import pandas as pd

# File path
file_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/Table_S6_updated.xlsx"

# Read the Excel sheet
df = pd.read_excel(file_path)

# Excel rows 2 to 41
rows_to_duplicate = df.iloc[0:38].copy()

# Duplicate 3 times
duplicated_rows = pd.concat([rows_to_duplicate] * 3, ignore_index=True)

# Append duplicated rows to the bottom
df_new = pd.concat([df, duplicated_rows], ignore_index=True)

# Save the new Excel file
output_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/EMT_duplicatsx.xlsx"

df_new.to_excel(output_path, index=False)

print("Done!")
print(f"Duplicated rows 2 to 41 three times and saved new file to:\n{output_path}")

Done!
Duplicated rows 2 to 41 three times and saved new file to:
/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected/EMT_duplicatsx.xlsx


In [1]:
import pandas as pd

# File path
file_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_expanded.xlsx"

# Read Excel file
df = pd.read_excel(file_path)

# Check for missing values
missing_rows = df[df.isnull().any(axis=1)]

# Print results
if missing_rows.empty:
    print("No missing values found in any row.")
else:
    print("Rows with missing values:\n")
    print(missing_rows)

    print("\nSummary of missing values per column:\n")
    print(df.isnull().sum())

No missing values found in any row.


In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# =========================
# Load original training data
# =========================
train_file_path = '/Users/eolanrew/Desktop/Filtered_dataset/Filtered_original_space.xlsx'
train_data = pd.read_excel(train_file_path)

# Ensure EMT column exists
if 'EMT' not in train_data.columns:
    raise ValueError("EMT column not found in the training data.")

# Columns used during training
columns_for_pca = [
    'EMT', 'Na', 'Al', 'Si', 'H2O',
    'Si_source_Fumed silica', 'Si_source_Ludox-AS40',
    'Si_source_Ludox-SM30', 'Si_source_Na2SiO3',
    'Si_source_TEOS', 'Pre-dissolution of Si',
    'Al_source_Al(OH)3', 'Al_source_NaAlO2',
    'Pre-dissolution of Al', 'Si/OH', 'Si/Al',
    'Time (h)', 'Temperature (°C)',
    'Si precursor sizs (nm)'
]

# Columns excluded from scaling
exclude_from_scaling = [
    'Si_source_Fumed silica', 'Si_source_Ludox-AS40',
    'Si_source_Ludox-SM30', 'Si_source_Na2SiO3',
    'Si_source_TEOS', 'Pre-dissolution of Si',
    'Al_source_Al(OH)3', 'Al_source_NaAlO2',
    'Pre-dissolution of Al'
]

# Columns to scale
columns_to_scale = [
    col for col in columns_for_pca
    if col not in exclude_from_scaling and col != 'EMT'
]

# =========================
# Fit scaler on original training data
# =========================
scaler = StandardScaler()

train_data_scaled = train_data[columns_for_pca].copy()
train_data_scaled[columns_to_scale] = scaler.fit_transform(
    train_data_scaled[columns_to_scale]
)

# =========================
# Load new file to transform
# =========================
new_file_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_expanded.xlsx'
new_data = pd.read_excel(new_file_path)

# Retain Source column
if 'Source' not in new_data.columns:
    raise ValueError("Source column not found in the new file.")

# Keep Source separately
source_column = new_data['Source']

# Select required columns
new_data_for_transform = new_data[columns_for_pca].copy()

# Apply ONLY transform using fitted scaler
new_data_for_transform[columns_to_scale] = scaler.transform(
    new_data_for_transform[columns_to_scale]
)

# Add Source column back
new_data_for_transform['Source'] = source_column

# =========================
# Save ONLY transformed new file
# =========================
output_file_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_duplicated_transformed.xlsx'

new_data_for_transform.to_excel(output_file_path, index=False)

print(f"Transformed file saved to:\n{output_file_path}")

Transformed file saved to:
/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_duplicated_transformed.xlsx


In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# =========================
# 1. TRAINING DATA (fit PCA)
# =========================
train_file = '/Users/eolanrew/Desktop/Filtered_dataset/scaled_data_with_EMT.xlsx'
train_data = pd.read_excel(train_file)

columns_for_pca = [
    'Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica', 'Si_source_Ludox-AS40',
    'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS',
    'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2',
    'Pre-dissolution of Al', 'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)',
    'Si precursor sizs (nm)'
]

X_train = train_data[columns_for_pca]

# Fit PCA on training data
pca = PCA(n_components=3)
train_pca = pca.fit_transform(X_train)


# =========================
# 2. NEW DATA (transform only)
# =========================
test_file = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_duplicated_transformed.xlsx'
test_data = pd.read_excel(test_file)

X_test = test_data[columns_for_pca]

# Apply SAME PCA transformation
test_pca = pca.transform(X_test)


# =========================
# 3. BUILD OUTPUT
# =========================
pca_columns = [f'PCA_Component_{i+1}' for i in range(3)]
pca_df = pd.DataFrame(test_pca, columns=pca_columns)

# keep Source column
source = test_data[['Source']]

# keep EMT column
emt = test_data[['EMT']]

# combine everything
final_data = pd.concat([source, emt, pca_df], axis=1)

# =========================
# 4. SAVE ONLY FINAL FILE
# =========================
output_file = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_PCA_transformed_only.xlsx'
final_data.to_excel(output_file, index=False)

print(f"\nSaved PCA-transformed file at: {output_file}")


Saved PCA-transformed file at: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_PCA_transformed_only.xlsx


In [4]:
import pandas as pd
import pickle

# Function to make predictions and save the updated file
def predict_and_save(model_path, pca_file_path, output_file_path):
    # Load the PCA-reduced data
    df = pd.read_excel(pca_file_path)
    
    # Ensure the necessary PCA columns are present
    required_columns = ['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    
    # Extract PCA components
    pca_data = df[required_columns].values
    
    # Load the model
    with open(model_path, 'rb') as file:
        model = pickle.load(file)
    
    # Make predictions (assume model outputs probabilities)
    probabilities = model.predict_proba(pca_data)[:, 1]
    
    # Add the probabilities as a new column to the DataFrame
    df['Probability'] = probabilities
    
    # Save the updated DataFrame to a new Excel file
    df.to_excel(output_file_path, index=False)
    print(f"File with probabilities saved to: {output_file_path}")

# Example usage
model_file_path = '/Users/eolanrew/Desktop/Filtered_dataset/xgb_pca.pkl'
pca_file_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_PCA_transformed_only.xlsx'
output_file_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_PCA_result.xlsx'

predict_and_save(model_file_path, pca_file_path, output_file_path)


File with probabilities saved to: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_PCA_result.xlsx


In [5]:
import pandas as pd
import pickle

# Load PCA model
pca_model_path = '/Users/eolanrew/Desktop/Filtered_dataset/pca_model.pkl'
with open(pca_model_path, 'rb') as file:
    pca = pickle.load(file)

# Load data
pca_results_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_PCA_result.xlsx'
df = pd.read_excel(pca_results_path)

# Extract PCA components
pca_components = df[['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3']].values

# Inverse transform
reconstructed_data = pca.inverse_transform(pca_components)

# Rebuild original feature columns
columns_for_pca = [
    'Na', 'Al', 'Si', 'H2O', 'Si_source_Fumed silica',
    'Si_source_Ludox-AS40', 'Si_source_Ludox-SM30',
    'Si_source_Na2SiO3', 'Si_source_TEOS',
    'Pre-dissolution of Si', 'Al_source_Al(OH)3',
    'Al_source_NaAlO2', 'Pre-dissolution of Al',
    'Si/OH', 'Si/Al', 'Time (h)', 'Temperature (°C)',
    'Si precursor sizs (nm)'
]

reconstructed_df = pd.DataFrame(reconstructed_data, columns=columns_for_pca)

# Drop PCA columns from original and replace with reconstructed ones
df_final = pd.concat(
    [df.drop(columns=['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3']),
     reconstructed_df],
    axis=1
)

# Save
output_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_PCA_inverse.xlsx'
df_final.to_excel(output_path, index=False)

print(f"Saved full dataset with reconstructed features at {output_path}")

Saved full dataset with reconstructed features at /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_PCA_inverse.xlsx


In [10]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib  # Import joblib for loading the scaler

# Load the reconstructed data
reconstructed_data_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_PCA_inverse_distance.xlsx'
reconstructed_data = pd.read_excel(reconstructed_data_path)

# Load the scaler model
scaler_file_path = '/Users/eolanrew/Desktop/Filtered_dataset/scaler_model.pkl'
scaler = joblib.load(scaler_file_path)

# Get the feature names used in the scaler
columns_to_scale = scaler.feature_names_in_  # This gets the names of features used in the scaler

# Inverse transform the scaled features in the reconstructed data
reconstructed_data[columns_to_scale] = scaler.inverse_transform(reconstructed_data[columns_to_scale])

# Save the final DataFrame with inverse scaled features to an Excel file
scaler_final_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_final_results.xlsx'
reconstructed_data.to_excel(scaler_final_path, index=False)

print(f"Inverse scaled DataFrame saved to {scaler_final_path}")


Inverse scaled DataFrame saved to /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Manuscript/EMT_final_results.xlsx


In [5]:
import pandas as pd
import numpy as np
from numpy.linalg import norm

# Load data
scaled_data = pd.read_excel('/Users/eolanrew/Desktop/Filtered_dataset/scaled_data_with_EMT.xlsx')
filtered_data = pd.read_excel('/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_duplicated_transformed.xlsx')

# Feature columns
cols = [
    'Na', 'Al', 'Si', 'H2O',
    'Si_source_Fumed silica', 'Si_source_Ludox-AS40',
    'Si_source_Ludox-SM30', 'Si_source_Na2SiO3', 'Si_source_TEOS',
    'Pre-dissolution of Si', 'Al_source_Al(OH)3', 'Al_source_NaAlO2',
    'Pre-dissolution of Al', 'Si/OH', 'Si/Al',
    'Time (h)', 'Temperature (°C)', 'Si precursor sizs (nm)'
]

scaled_features = scaled_data[cols].values
filtered_features = filtered_data[cols].values

min_distances = []
closest_indices = []

# For each filtered row, find closest in scaled_data
for i in range(filtered_features.shape[0]):
    row = filtered_features[i]

    # compute all distances to scaled_data
    distances = np.linalg.norm(scaled_features - row, axis=1)

    min_idx = np.argmin(distances)
    min_dist = distances[min_idx]

    min_distances.append(min_dist)
    closest_indices.append(min_idx)

# Add results to filtered_data
filtered_data['closest_scaled_row'] = closest_indices
filtered_data['min_euclidean_distance'] = min_distances

# Save
output_path = '/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/EMT_filtered_with_distances.xlsx'
filtered_data.to_excel(output_path, index=False)

print(filtered_data[['closest_scaled_row', 'min_euclidean_distance']].head())

   closest_scaled_row  min_euclidean_distance
0                  30               38.594936
1                  30               38.607889
2                  30               38.607889
3                  30               38.620838
4                  30               38.595515


In [6]:
import pandas as pd

# File paths
file_base = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/corrected_1/Table_S6_expanded.xlsx"
file_prob = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected_1/EMT_PCA_result.xlsx"
file_dist = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected_1/EMT_filtered_with_distances.xlsx"

# Load files
df_base = pd.read_excel(file_base)
df_prob = pd.read_excel(file_prob)
df_dist = pd.read_excel(file_dist)

# Copy columns
df_base["Probability"] = df_prob["Probability"]
df_base["min_euclidean_distance"] = df_dist["min_euclidean_distance"]

# Save result
output_path = "/Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected_1/EMT_Finals.xlsx"
df_base.to_excel(output_path, index=False)

print("Done. File saved to:", output_path)

Done. File saved to: /Users/eolanrew/Desktop/Qlearning/Inference/KV Cache/Corrected_1/EMT_Finals.xlsx
